# Demos

## Set environment
```
./setup
export AGENT_API_KEY=...        # never commit this secret
```

In [ ]:
# This needs AGENT_API_KEY (or AGENT_BASE_URL to override)
import asyncio
from agent import Agent
from pathlib import Path

# Create content
notes_path = Path("notes.md")
notes_path.write_text("""
# Meeting notes
- The agent harness exposes a single `Agent` class.
- Sessions remember their conversation between runs.
- Every tool call is streamed with its arguments and outcome.
"""
)

# Template prompt
TEMPLATE = """
Summarise the following notes as bullet points:
{{ config.input }}
"""

# Create the harness
harness = Agent()

# Invoke the harness
result = asyncio.run(harness.run(TEMPLATE, input=notes_path))

# Display the result
print("done:", "ok" if result.ok else result.error)
print("error:", result.error)
print("output:", result.output)
print("session:", result.session_id)
print("tool calls:", [call.signature() for call in result.tools])


In [ ]:
# The session remembers its conversation: passing the same session id runs
# with the earlier turns replayed, so the model knows what it did before.

harness_2 = Agent()

followup = asyncio.run(harness_2.run("""
How many bullet points did you produce, and what was the first one?
{{ config.input }}
""", 
input="notes.md", session=result.session_id))

print("output:", followup.output)
print("session:", followup.session_id, "(same as before)" if followup.session_id == result.session_id else "(new!)")
